# 06 · Crash-risk model (F3)

Expected injury crashes per H3 cell × weekday × hour. Train on 2024 (features from earlier years only), test on 2025. LightGBM Poisson v1 (history + OpenStreetMap road features) vs v0 (history only) and two baselines; cell-block bootstrap for the differences. Runs on CPU in seconds (zero-cost policy: no paid compute).

OSM features: run `python -m bhtraffic.geo.osm` once (or the next cell); without them the model falls back to v0.

In [ ]:
# Setup: works on Colab (Drive + private repo) and locally (repo checkout).
import sys, os
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.environ["BHTRAFFIC_DATA_ROOT"] = "/content/drive/MyDrive/transito_bh"
    token = userdata.get("GH_TOKEN")  # Colab Secrets: fine-grained token, read-only on the repo
    # Clone (not pip-from-git): the package resolves data/ and paper/ relative to the repo root.
    url = f"https://{token}@github.com/Dias0202/Transito_bh.git"
    if os.path.isdir("/content/Transito_bh"):
        !git -C /content/Transito_bh pull -q {url} main
    else:
        !git clone -q {url} /content/Transito_bh
        !git -C /content/Transito_bh remote set-url origin https://github.com/Dias0202/Transito_bh.git
    %pip -q install -e "/content/Transito_bh[geo,viz,ml]"
    sys.path.insert(0, "/content/Transito_bh/src")
    os.chdir("/content/Transito_bh/notebooks")
else:
    from pathlib import Path
    sys.path.insert(0, str(Path.cwd().parent / "src"))

In [ ]:
# Crash series 2011 -> last complete month (yearly + monthly PBH datasets). Built once; free downloads.
from bhtraffic.config import processed_dir
if not (processed_dir('sinistros') / 'ocorrencias_2011_2026.parquet').exists():
    from bhtraffic.ingest import sinistros, recentes
    if not (processed_dir('sinistros') / 'ocorrencias.parquet').exists():
        sinistros.build()
    recentes.build()

In [ ]:
from bhtraffic.config import processed_dir
if not (processed_dir('osm') / 'h3_features.parquet').exists():
    from bhtraffic.geo import osm
    osm.build()  # free Overpass download, cached; a few minutes

In [ ]:
import json
from bhtraffic.models import risk
res = risk.fit_eval()
pd_metrics = {k: v for k, v in res.metrics.items() if isinstance(v, dict)}
import pandas as pd
pd.DataFrame(pd_metrics).round(3)

In [ ]:
{'v1_minus_cellxprofile': risk.bootstrap_diff(res.test, 'model', 'cellxprofile'),
 'v1_minus_v0': risk.bootstrap_diff(res.test, 'model', 'v0')}

In [ ]:
imp, _, _ = risk.explain(res)
imp.round(3)

In [ ]:
from bhtraffic.viz import figures as F
res.metrics['shap_mean_abs'] = imp.to_dict()
for lang in ('pt', 'en'):
    print(F.risk_capture(res.test, res.metrics['test_year'], lang), F.risk_shap(res.metrics['shap_mean_abs'], lang))

**Reading.** v1, v0 and the cell-rate × hourly-profile baseline are statistically tied (bootstrap CIs include 0): with crash history, risk is almost separable into *where* × *when*, and OSM road features add nothing the history does not already carry. The next lever is exposure (radar volumes) and time-varying conditions (rain, holidays: see `bhtraffic.models.temporal`).